# 02 — Analyse de la qualité des données

## Objectif

Ce notebook évalue la qualité du dataset avant la construction
du pipeline ETL.

Les contrôles portent sur :

- les valeurs manquantes ;
- les doublons ;
- les identifiants ;
- les types ;
- les catégories ;
- les valeurs incohérentes ;
- les colonnes constantes ;
- les colonnes fortement incomplètes.

In [ ]:
from pathlib import Path

import polars as pl

In [ ]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "source" / "diabetic_data.csv"

In [ ]:
df_raw = pl.read_csv(
    DATA_PATH,
    infer_schema_length=10000,
)

In [ ]:
df = pl.read_csv(
    DATA_PATH,
    null_values=["?", ""],
    infer_schema_length=10000,
)

In [ ]:
null_summary = pl.DataFrame(
    {
        "column": df.columns,
        "null_count": [
            df.select(pl.col(column).null_count()).item()
            for column in df.columns
        ],
    }
).with_columns(
    (
        pl.col("null_count") / df.height * 100
    ).round(2).alias("null_percentage")
).sort("null_percentage", descending=True)

null_summary

In [ ]:
null_summary.filter(
    pl.col("null_count") > 0
)

In [ ]:
null_summary.with_columns(
    pl.when(pl.col("null_percentage") == 0)
    .then(pl.lit("Aucune valeur manquante"))
    .when(pl.col("null_percentage") < 5)
    .then(pl.lit("Faible"))
    .when(pl.col("null_percentage") < 20)
    .then(pl.lit("Modéré"))
    .when(pl.col("null_percentage") < 50)
    .then(pl.lit("Élevé"))
    .otherwise(pl.lit("Critique"))
    .alias("severity")
)

In [ ]:
question_mark_summary = []

for column in df_raw.columns:
    if df_raw.schema[column] == pl.String:
        count = df_raw.select(
            (pl.col(column) == "?").sum()
        ).item()

        if count > 0:
            question_mark_summary.append(
                {
                    "column": column,
                    "question_mark_count": count,
                    "percentage": round(count / df_raw.height * 100, 2),
                }
            )

question_mark_df = pl.DataFrame(question_mark_summary).sort(
    "percentage",
    descending=True,
)

question_mark_df

In [ ]:
duplicate_row_count = df.height - df.unique().height

print(f"Nombre de lignes dupliquées : {duplicate_row_count}")

In [ ]:
df_pd = df.to_pandas()

duplicate_row_count = df_pd.duplicated().sum()

print(duplicate_row_count)

In [ ]:
encounter_duplicates = (
    df.group_by("encounter_id")
    .agg(pl.len().alias("count"))
    .filter(pl.col("count") > 1)
)

encounter_duplicates

In [ ]:
print(
    f"Nombre d'encounter_id dupliqués : "
    f"{encounter_duplicates.height}"
)

In [ ]:
patient_duplicates = (
    df.group_by("patient_nbr")
    .agg(pl.len().alias("encounter_count"))
    .filter(pl.col("encounter_count") > 1)
    .sort("encounter_count", descending=True)
)

patient_duplicates.head(20)

In [ ]:
df.filter(
    (pl.col("time_in_hospital") <= 0)
    | pl.col("time_in_hospital").is_null()
)

In [ ]:
df.filter(
    pl.col("num_medications") < 0
)

In [ ]:
df.filter(
    pl.col("number_diagnoses") < 0
)

In [ ]:
visit_columns = [
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
]

for column in visit_columns:
    invalid_count = df.filter(
        pl.col(column) < 0
    ).height

    print(f"{column} : {invalid_count} valeurs négatives")

In [ ]:
expected_readmitted_values = {"<30", ">30", "NO"}

actual_readmitted_values = set(
    df.select("readmitted")
    .drop_nulls()
    .to_series()
    .to_list()
)

unexpected_readmitted_values = (
    actual_readmitted_values - expected_readmitted_values
)

print(unexpected_readmitted_values)

In [ ]:
df.select(
    pl.col("gender").value_counts(sort=True)
)

In [ ]:
expected_medication_values = {
    "No",
    "Steady",
    "Up",
    "Down",
}

In [ ]:
medication_quality_results = []

for column in existing_medication_columns:
    values = set(
        df.select(column)
        .drop_nulls()
        .to_series()
        .to_list()
    )

    unexpected_values = values - expected_medication_values

    medication_quality_results.append(
        {
            "column": column,
            "unexpected_values": str(unexpected_values),
            "is_valid": len(unexpected_values) == 0,
        }
    )

pl.DataFrame(medication_quality_results)

In [ ]:
high_missing_columns = null_summary.filter(
    pl.col("null_percentage") >= 40
)

high_missing_columns

In [ ]:
quality_summary = pl.DataFrame(
    {
        "column": df.columns,
        "dtype": [str(df.schema[column]) for column in df.columns],
        "null_count": [
            df.select(pl.col(column).null_count()).item()
            for column in df.columns
        ],
        "unique_count": [
            df.select(pl.col(column).n_unique()).item()
            for column in df.columns
        ],
    }
).with_columns(
    (
        pl.col("null_count") / df.height * 100
    ).round(2).alias("null_percentage")
).sort("null_percentage", descending=True)

quality_summary

In [ ]:
REPORT_OUTPUT = (
    PROJECT_ROOT
    / "docs"
    / "data_quality_summary.csv"
)

quality_summary.write_csv(REPORT_OUTPUT)

print(f"Rapport exporté vers : {REPORT_OUTPUT}")

# Conclusion de l’analyse qualité

L’analyse a permis d’identifier les principales anomalies du dataset :

- valeurs manquantes réelles ou représentées par `?` ;
- colonnes fortement incomplètes ;
- variables catégorielles avec valeurs inconnues ;
- codes nécessitant un mapping ;
- diagnostics nécessitant une normalisation ;
- déséquilibre possible de la variable cible ;
- présence de patients avec plusieurs séjours.

Ces observations serviront à concevoir les règles de validation,
la zone de quarantaine et le futur pipeline ETL.